# 🤖 LLM Agents com HuggingFace — smolagents
### Atividade #8 – Indo Além

---

## 🧠 O que é um LLM Agent?

Um **LLM Agent** (Agente de Modelo de Linguagem) é um sistema em que um **modelo de linguagem** (LLM) atua como o "cérebro" de um fluxo de decisões. Em vez de apenas gerar texto, o agente:

1. **Recebe** uma tarefa em linguagem natural
2. **Decide** quais ferramentas usar (calculadora, busca, API, etc.)
3. **Executa** as ferramentas e observa os resultados
4. **Itera** até chegar na resposta final

Neste notebook usaremos a biblioteca **`smolagents`** do HuggingFace para construir um agente simples capaz de:
- Realizar cálculos matemáticos
- Contar palavras em um texto
- Converter temperaturas (Celsius ↔ Fahrenheit)

---

## 📦 Passo 1: Instalação das Bibliotecas

In [4]:
# Instalamos o smolagents (biblioteca oficial de agentes do HuggingFace)
# e o huggingface_hub para autenticação com a API
!pip install smolagents huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 🔑 Passo 2: Autenticação no HuggingFace

Para usar os modelos via Inference API, precisamos de um token do HuggingFace.

👉 Crie sua conta gratuita em: https://huggingface.co  
👉 Gere seu token em: https://huggingface.co/settings/tokens

In [5]:
import os

# Cole seu token do HuggingFace aqui
# Você pode criar um token gratuito em: https://huggingface.co/settings/tokens
os.environ["HF_TOKEN"] = "Minha_Chave"  # ← substitua pelo seu token

# Confirmação
print("✅ Token configurado!")

✅ Token configurado!


## 🛠️ Passo 3: Importações

In [6]:
# smolagents: biblioteca oficial de agentes do HuggingFace
from smolagents import CodeAgent, tool, InferenceClientModel

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


## 🔧 Passo 4: Criando as Ferramentas (Tools)

**Ferramentas** são funções Python que o agente pode chamar para executar tarefas específicas.  
O decorador `@tool` informa ao agente que aquela função está disponível para uso.  

> ⚠️ O **docstring** (texto entre `"""`) é obrigatório: é por ele que o agente entende o que a ferramenta faz!

In [7]:
@tool
def calculadora(expressao: str) -> str:
    """
    Avalia uma expressão matemática e retorna o resultado.
    Use para somar, subtrair, multiplicar, dividir ou calcular potências.
    Exemplo de entrada: '25 * 4 + 10'

    Args:
        expressao: a expressão matemática como string, por exemplo '2 + 2' ou '10 * 5'
    """
    try:
        resultado = eval(expressao)  # avalia a expressão matematicamente
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Erro ao calcular: {str(e)}"


@tool
def contar_palavras(texto: str) -> str:
    """
    Conta o número de palavras em um texto.
    Use quando precisar saber quantas palavras tem uma frase ou parágrafo.

    Args:
        texto: o texto do qual se deseja contar as palavras
    """
    palavras = texto.split()           # separa as palavras por espaço
    total = len(palavras)              # conta quantas existem
    return f"O texto possui {total} palavra(s)."


@tool
def celsius_para_fahrenheit(celsius: float) -> str:
    """
    Converte uma temperatura de Celsius para Fahrenheit.
    Use quando precisar converter temperaturas entre as duas escalas.

    Args:
        celsius: a temperatura em graus Celsius (número)
    """
    fahrenheit = (celsius * 9 / 5) + 32   # fórmula de conversão
    return f"{celsius}°C = {fahrenheit:.1f}°F"


print("✅ Ferramentas criadas:")
print("   🔢 calculadora")
print("   📝 contar_palavras")
print("   🌡️  celsius_para_fahrenheit")

✅ Ferramentas criadas:
   🔢 calculadora
   📝 contar_palavras
   🌡️  celsius_para_fahrenheit


## 🤖 Passo 5: Criando o Agente

Agora conectamos tudo: o **modelo de linguagem** + as **ferramentas** = nosso agente!

- `InferenceClientModel`: conecta com o modelo via API do HuggingFace (sem precisar rodar localmente)
- `CodeAgent`: tipo de agente que gera código Python para chamar as ferramentas

In [9]:
# Escolhemos o modelo que vai "pensar" por nosso agente
# Qwen2.5-72B-Instruct é um LLM poderoso disponível gratuitamente no HuggingFace
modelo = InferenceClientModel(
    model_id="Qwen/Qwen2.5-72B-Instruct",
    token=os.environ["HF_TOKEN"]
)

# Criamos o agente, passando o modelo e as ferramentas disponíveis
agente = CodeAgent(
    tools=[calculadora, contar_palavras, celsius_para_fahrenheit],
    model=modelo
)

print("✅ Agente criado e pronto para uso!")

✅ Agente criado e pronto para uso!


## 🚀 Passo 6: Testando o Agente

Agora vamos ver o agente em ação! Ele irá:
1. Ler nossa pergunta
2. Decidir quais ferramentas usar
3. Executar as ferramentas
4. Combinar os resultados para responder

In [10]:
# ───────────────────────────────────────────────
# Tarefa 1: Problema que exige calculadora
# ───────────────────────────────────────────────
print("=" * 60)
print("🧮 TAREFA 1: Cálculo matemático")
print("=" * 60)

resposta1 = agente.run(
    "Se eu tenho 15 laranjas e compro mais 8, depois dou metade para um amigo, com quantas fico?"
)

print("\n📌 Resposta final:", resposta1)

🧮 TAREFA 1: Cálculo matemático


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Se eu tenho 15 laranjas e compro mais 8, depois dou metade para um amigo, com quantas fico?                     │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  total_oranges = calculadora('15 + 8')                                                                            
  oranges_left = calculadora(f'{total_oranges} / 2')                                                               
  print(oranges_left)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 6 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(11)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 11

[Step 6: Duration 2.64 seconds| Input tokens: 16,766 | Output tokens: 692]


📌 Resposta final: 11


In [11]:
# ───────────────────────────────────────────────
# Tarefa 2: Problema que exige contagem de palavras
# ───────────────────────────────────────────────
print("=" * 60)
print("📝 TAREFA 2: Contagem de palavras")
print("=" * 60)

frase = "A inteligência artificial está transformando o mundo de maneiras incríveis e surpreendentes"
resposta2 = agente.run(
    f"Quantas palavras tem a seguinte frase? '{frase}' E qual é o quadrado desse número?"
)

print("\n📌 Resposta final:", resposta2)

📝 TAREFA 2: Contagem de palavras


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Quantas palavras tem a seguinte frase? 'A inteligência artificial está transformando o mundo de maneiras        │
│ incríveis e surpreendentes' E qual é o quadrado desse número?                                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  sentence = 'A inteligência artificial está transformando o mundo de maneiras incríveis e surpreendentes'         
  word_count = int(contar_palavras(sentence))                                                                      
  square_of_word_count = calculadora(f'{word_count} ** 2')                                                         
  print(f'The sentence has {word_count} words.')                                                                   
  print(f'The square of the word count is {square_of_word_count}.')                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'word_count = int(contar_palavras(sentence))' due to: ValueError: invalid literal for
int() with base 10: 'O texto possui 12 palavra(s).'

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(f'The sentence has {word_count} words and the square of the word count is                           
  {square_of_word_count}.')                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The sentence has 12 words and the square of the word count is Resultado: 144.

[Step 3: Duration 5.06 seconds| Input tokens: 7,833 | Output tokens: 332]


📌 Resposta final: The sentence has 12 words and the square of the word count is Resultado: 144.


In [12]:
# ───────────────────────────────────────────────
# Tarefa 3: Problema que combina múltiplas ferramentas
# ───────────────────────────────────────────────
print("=" * 60)
print("🌡️  TAREFA 3: Conversão de temperatura + cálculo")
print("=" * 60)

resposta3 = agente.run(
    "A temperatura em São Paulo hoje é 28°C. Converta para Fahrenheit. "
    "Depois calcule a diferença entre esse valor em Fahrenheit e 100."
)

print("\n📌 Resposta final:", resposta3)

🌡️  TAREFA 3: Conversão de temperatura + cálculo


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ A temperatura em São Paulo hoje é 28°C. Converta para Fahrenheit. Depois calcule a diferença entre esse valor   │
│ em Fahrenheit e 100.                                                                                            │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  temp_fahrenheit = celsius_para_fahrenheit(28)                                                                    
  print("Temp_fahrenheit:", temp_fahrenheit)                                                                       
  difference = calculadora(f'{temp_fahrenheit} - 100')                                                             
  print("Difference:", difference)                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Temp_fahrenheit: 28°C = 82.4°F
Difference: Erro ao calcular: invalid character '°' (U+00B0) (<string>, line 1)

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(-17.6)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: -17.6

[Step 4: Duration 2.54 seconds| Input tokens: 10,783 | Output tokens: 361]


📌 Resposta final: -17.6


## 📊 Passo 7: Entendendo o que aconteceu

Veja como o agente se comportou em cada tarefa:

| Tarefa | Ferramentas usadas | Comportamento do agente |
|--------|-------------------|-------------------------|
| Laranjas | `calculadora` | Identificou que precisava fazer (15+8)/2 |
| Contagem + quadrado | `contar_palavras` + `calculadora` | Encadeou 2 ferramentas em sequência |
| Temperatura | `celsius_para_fahrenheit` + `calculadora` | Usou 2 ferramentas com dependência entre elas |

Isso demonstra o poder dos **LLM Agents**: o modelo **raciocina** e **orquestra** as ferramentas automaticamente, sem precisarmos dizer explicitamente qual ferramenta usar!

## 🏁 Conclusão

Neste notebook aprendemos a:

✅ Entender o conceito de **LLM Agents**  
✅ Criar **ferramentas personalizadas** com o decorador `@tool`  
✅ Usar o modelo **Qwen2.5-72B** via HuggingFace Inference API  
✅ Construir um **CodeAgent** capaz de raciocinar e encadear ferramentas  
✅ Ver o agente resolver problemas reais de forma autônoma  

---

## 📚 Referências

- [smolagents – Documentação Oficial](https://huggingface.co/docs/smolagents)
- [HuggingFace – Plataforma de Modelos](https://huggingface.co)
- [Qwen2.5-72B-Instruct no HuggingFace](https://huggingface.co/Qwen/Qwen2.5-72B-Instruct)
- [Guia: Building Agents with smolagents](https://huggingface.co/blog/smolagents)
- [HuggingFace Inference API](https://huggingface.co/docs/api-inference)